# Composizione del dataframe (data analysis)
In questo notebook vengono modellati i dati ottenuti dal dataset [Italian Serie A (football)](https://datahub.io/football/italian-serie-a), a partire dalla stagione 2020/2021 fino alla 2025/2026. A seguito indichiamo i campi che vengono mantenuti e quelli eliminati, elencandone le motivazioni.

## Installazione delle librerie
Per l'analisi dei dati, utilizziamo le seguenti librerie:
- Pandas
- Numpy
che sono installate nell'ambiente virtuale che stiamo utilizzando, e che vengono importate nel seguente blocco

In [ ]:
import numpy as np
import pandas as pd

## Importazione dataset
Nella seguente cella vengono importati i dataset che utilizziamo per l'analisi dei dati. Essendo separati li uniamo tramite la funzione di pandas "concat", e successivamente
li andiamo ad analizzare 

In [11]:
import glob
import os
import pandas as pd

# Indico la cartella nella quale sono contenuti i dataset
path_cartella = 'static/dataset'

csv_files = os.path.join(path_cartella, "*.csv")
all_files = glob.glob(csv_files)

# Creo un array di dataframe e li concateno
lista_dataframe = [pd.read_csv(file) for file in all_files]
raw_dataset = pd.concat(lista_dataframe, axis=0, ignore_index=True)

# Converto la colonna 'Date' a un formato data
raw_dataset['Date'] = pd.to_datetime(raw_dataset['Date'])

# Ordino l'intero dataset usando la colonna 'Date'
raw_dataset = raw_dataset.sort_values(by='Date', ascending=False)

# Resetto gli indici
raw_dataset = raw_dataset.reset_index(drop=True)

raw_dataset.head()


,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,...,HST,AST,HF,AF,HC,AC,HY,AY,HR,AR
0,2026-05-24,Lecce,Genoa,1,0,H,1,0,H,NaN,...,4,1,11,6,1,6,2,0,0,0
1,2026-05-24,Cremonese,Como,1,4,A,0,1,A,NaN,...,1,6,14,12,2,4,4,2,3,0
2,2026-05-24,Parma,Sassuolo,1,0,H,0,0,D,NaN,...,3,1,10,14,1,1,2,2,0,0
3,2026-05-24,Napoli,Udinese,1,0,H,1,0,H,NaN,...,6,3,3,13,3,3,0,3,0,1
4,2026-05-24,Milan,Cagliari,1,2,A,1,1,D,NaN,...,3,10,8,9,5,5,2,2,0,0


## Rimozione dati aggiuntivi

Rimuoviamo i dati meno rilevati ai fini dell nostra analisi dal dataset. Focalizzandoci sui seguenti 
- Data partita
- Squadra locale ed ospite
- FTHG (Full Time Home Goals) FTAG (Full Time Away Goals)
- FTR (Full Time Result)
- HST (Home Shots on Target) AST (Away Shots on Target) 

In [12]:
new_index = [
    'Date',
    'HomeTeam',
    'AwayTeam',     
    'FTHG',
    'FTAG',
    'HST',
    'AST',
    'FTR'
]

clean_dataset = raw_dataset[new_index]
clean_dataset.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,HST,AST,FTR
0,2026-05-24,Lecce,Genoa,1,0,4,1,H
1,2026-05-24,Cremonese,Como,1,4,1,6,A
2,2026-05-24,Parma,Sassuolo,1,0,3,1,H
3,2026-05-24,Napoli,Udinese,1,0,6,3,H
4,2026-05-24,Milan,Cagliari,1,2,3,10,A


## Creazione di nuove colonne
Adesso tramire il dataframe rifinito e i suoi attributi, ne creiamo di nuovi, derviati, da utilizzare per l'allenamento del modello
che andremmo successivamente a generare. In particolare aggiungiamo 5 nuove categorie, ovvero:
- Stagione del campionato della riga di riferimento, in formato anno1-anno2 (es. 2023-2024);
- Media delle vittore delle ultime 5 partite;
- Media dei goal segnati e subiti nelle ultime 5 partite;
- Media del rapporto $\frac{GOAL}{TIRI\_IN\_PORTA}$ delle ultime 5 partite;
- **Home advantage**, ovvero la percentuale delle vittorie in casa nelle ultime 5 partite.

### Stagione del campionato
Questo campo è facilmente derivabile dalla data in cui si disputa la partita: se è svolta prima di Luglio dell'anno X, allora la stagione sarà (X-1)-(X), caso contrario sarà (X)-X(X+1).

In [15]:
# Ottengo gli anni della stagione in base alla data
years = clean_dataset['Date'].dt.year
months = clean_dataset['Date'].dt.month

left_year = years.where(months > 7, years - 1)
right_year = left_year + 1

# Aggiungo la categoria "Season" al dataset
season_dataset = clean_dataset.copy()
season_dataset["Season"] = left_year.astype(str) + "-" + right_year.astype(str)

season_dataset.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,HST,AST,FTR,Season
0,2026-05-24,Lecce,Genoa,1,0,4,1,H,2025-2026
1,2026-05-24,Cremonese,Como,1,4,1,6,A,2025-2026
2,2026-05-24,Parma,Sassuolo,1,0,3,1,H,2025-2026
3,2026-05-24,Napoli,Udinese,1,0,6,3,H,2025-2026
4,2026-05-24,Milan,Cagliari,1,2,3,10,A,2025-2026
